In [1]:
# Client Setup
from dotenv import load_dotenv
import voyageai

load_dotenv()

client = voyageai.Client()

In [2]:
# Chunk by section
import re


def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)

In [3]:
# Embedding Generation
def generate_embedding(chunks, model="voyage-3-large", input_type="query"):
    is_list = isinstance(chunks, list)
    input = chunks if is_list else [chunks]
    result = client.embed(input, model=model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]

In [4]:
# VectorIndex implementation
import math
from typing import Optional, Any, List, Dict, Tuple


class VectorIndex:
    def __init__(
        self,
        distance_metric: str = "cosine",
        embedding_fn=None,
    ):
        self.vectors: List[List[float]] = []
        self.documents: List[Dict[str, Any]] = []
        self._vector_dim: Optional[int] = None
        if distance_metric not in ["cosine", "euclidean"]:
            raise ValueError("distance_metric must be 'cosine' or 'euclidean'")
        self._distance_metric = distance_metric
        self._embedding_fn = embedding_fn

    def add_document(self, document: Dict[str, Any]):
        if not self._embedding_fn:
            raise ValueError(
                "Embedding function not provided during initialization."
            )
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        content = document["content"]
        if not isinstance(content, str):
            raise TypeError("Document 'content' must be a string.")

        vector = self._embedding_fn(content)
        self.add_vector(vector=vector, document=document)

    def search(
        self, query: Any, k: int = 1
    ) -> List[Tuple[Dict[str, Any], float]]:
        if not self.vectors:
            return []

        if isinstance(query, str):
            if not self._embedding_fn:
                raise ValueError(
                    "Embedding function not provided for string query."
                )
            query_vector = self._embedding_fn(query)
        elif isinstance(query, list) and all(
            isinstance(x, (int, float)) for x in query
        ):
            query_vector = query
        else:
            raise TypeError(
                "Query must be either a string or a list of numbers."
            )

        if self._vector_dim is None:
            return []

        if len(query_vector) != self._vector_dim:
            raise ValueError(
                f"Query vector dimension mismatch. Expected {self._vector_dim}, got {len(query_vector)}"
            )

        if k <= 0:
            raise ValueError("k must be a positive integer.")

        if self._distance_metric == "cosine":
            dist_func = self._cosine_distance
        else:
            dist_func = self._euclidean_distance

        distances = []
        for i, stored_vector in enumerate(self.vectors):
            distance = dist_func(query_vector, stored_vector)
            distances.append((distance, self.documents[i]))

        distances.sort(key=lambda item: item[0])

        return [(doc, dist) for dist, doc in distances[:k]]

    def add_vector(self, vector, document: Dict[str, Any]):
        if not isinstance(vector, list) or not all(
            isinstance(x, (int, float)) for x in vector
        ):
            raise TypeError("Vector must be a list of numbers.")
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        if not self.vectors:
            self._vector_dim = len(vector)
        elif len(vector) != self._vector_dim:
            raise ValueError(
                f"Inconsistent vector dimension. Expected {self._vector_dim}, got {len(vector)}"
            )

        self.vectors.append(list(vector))
        self.documents.append(document)

    def _euclidean_distance(
        self, vec1: List[float], vec2: List[float]
    ) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return math.sqrt(sum((p - q) ** 2 for p, q in zip(vec1, vec2)))

    def _dot_product(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return sum(p * q for p, q in zip(vec1, vec2))

    def _magnitude(self, vec: List[float]) -> float:
        return math.sqrt(sum(x * x for x in vec))

    def _cosine_distance(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")

        mag1 = self._magnitude(vec1)
        mag2 = self._magnitude(vec2)

        if mag1 == 0 and mag2 == 0:
            return 0.0
        elif mag1 == 0 or mag2 == 0:
            return 1.0

        dot_prod = self._dot_product(vec1, vec2)
        cosine_similarity = dot_prod / (mag1 * mag2)
        cosine_similarity = max(-1.0, min(1.0, cosine_similarity))

        return 1.0 - cosine_similarity

    def __len__(self) -> int:
        return len(self.vectors)

    def __repr__(self) -> str:
        has_embed_fn = "Yes" if self._embedding_fn else "No"
        return f"VectorIndex(count={len(self)}, dim={self._vector_dim}, metric='{self._distance_metric}', has_embedding_fn='{has_embed_fn}')"

In [5]:
with open("./report.md", "r") as f:
    text = f.read()

In [8]:
# 1. Chunk the text by section
chunks = chunk_by_section(text)

chunks[4]

'Section 1: Medical Research - Understanding XDR-471 Syndrome\n\nThis year saw significant strides in our understanding of XDR-471 syndrome, a rare neurodegenerative condition previously hampered by diagnostic ambiguity. The team focused on correlating clinical presentations with specific genetic markers, particularly variations within the Gene LOC73b region. Analysis of patient cohort data (Cohort ID: XDR-EU-03) revealed a statistically significant link between symptom severity and marker expression levels, measured via quantitative PCR assays. Preliminary work on a novel diagnostic biomarker panel (Panel ID: XDR-BioMk-v2) shows promise, achieving >85% sensitivity in early validation sets. However, specificity remains a challenge requiring further refinement. Ongoing efforts under Trial ID: XDR-TR002 are exploring targeted therapeutic interventions based on these findings. These results provide a much-needed foundation for future clinical strategies, though the resource implications h

In [10]:
# 2. Generate embeddings for each chunk
embeddings = generate_embedding(chunks)

embeddings[1]

[-0.057809364050626755,
 0.04144780710339546,
 -0.029665851965546608,
 0.005685810931026936,
 -0.015950480476021767,
 0.00938964169472456,
 -0.027458779513835907,
 -0.012747341766953468,
 0.007358611095696688,
 -0.03981513902544975,
 0.006363443564623594,
 0.05183406546711922,
 -0.01950354315340519,
 0.004429390653967857,
 -0.016891850158572197,
 0.005464817862957716,
 0.029045552015304565,
 0.10027327388525009,
 -0.018263349309563637,
 -0.004620873369276524,
 0.039197586476802826,
 -0.06476392596960068,
 0.02487340196967125,
 -0.019138207659125328,
 -0.025813311338424683,
 0.0017593946540728211,
 0.015935024246573448,
 0.056154876947402954,
 0.006169011350721121,
 -0.03549967333674431,
 0.005048239137977362,
 0.0071405079215765,
 -0.07075135409832001,
 0.07170624285936356,
 0.01565484330058098,
 0.03281824290752411,
 -0.01572291925549507,
 -0.04569931700825691,
 -0.022907638922333717,
 -0.07204623520374298,
 -0.022798404097557068,
 0.016048170626163483,
 0.01386447623372078,
 -0.06460

In [ ]:
# 3. Create a vector store and add each embedding to it
# Note: converted to a bulk operation to avoid rate limiting errors from VoyageAI
store = VectorIndex()
# store.add_documents([{"content": chunk} for chunk in chunks])

for embedding, chunk in zip(embeddings, chunks):
    # INclude the original chun ktext alongside each ambedding 
    store.add_vector(embedding, {"content": chunk})

In [13]:
# 4. Some time later, a user will ask a question. Generate an embedding for it
user_embedding = generate_embedding("What did the software engineering department do last year?")

In [17]:
# 5. Search the store with the embedding, find the 2 most relevant chunks
results = store.search(user_embedding, 2)

for doc, distance in results:
    print(f"Distance between query and doc: {distance}.\n Doc content: {doc} \n")

Distance between query and doc: 0.4712959694004407.
 Doc content: {'content': 'Section 2: Software Engineering - Project Phoenix Stability Enhancements\n\nThe Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems underpinning Project Phoenix. Recurring issues, particularly `ERR_MEM_ALLOC_FAIL_0x8007000E` during peak loads and `TIMEOUT_QUERY_DB_0xDEADBEEF` affecting data retrieval operations, were prioritized, at a cost of INC-2023-Q4-011. Root cause analysis pointed towards inefficiencies in the primary data caching algorithm and suboptimal database indexing strategies. The deployment of a patch addressed the memory allocation error, resulting in a measured 40% reduction in critical failures under simulated stress tests during Q4 2024 (Test Case ID: INC-2023-Q4-011). Further refactoring of the query module, scheduled for the next release cycle, aims to resolve the timeout issue. These findings underscore the importanc